In [1]:
# ------------------- TASK 01 ------------------- #

 
from ortools.sat.python import cp_model
import math

grid = [[0]*5 for _ in range(5)]

start = (1, 1)
goal = (4, 4)

moves = [(-1, -1), (-1, 1), (1, -1), (1, 1)]

model = cp_model.CpModel()

MAX_STEPS = 7 

x = [model.NewIntVar(0, 4, f'x{t}') for t in range(MAX_STEPS)]
y = [model.NewIntVar(0, 4, f'y{t}') for t in range(MAX_STEPS)]

step_used = [model.NewBoolVar(f'step_used_{t}') for t in range(MAX_STEPS)]

model.Add(x[0] == start[0])
model.Add(y[0] == start[1])
model.Add(step_used[0] == 1)

goal_reached = [model.NewBoolVar(f'goal_at_{t}') for t in range(MAX_STEPS)]
for t in range(MAX_STEPS):
    model.Add(x[t] == goal[0]).OnlyEnforceIf(goal_reached[t])
    model.Add(y[t] == goal[1]).OnlyEnforceIf(goal_reached[t])
    model.Add(step_used[t] == 1).OnlyEnforceIf(goal_reached[t])
model.AddBoolOr(goal_reached)

for t in range(MAX_STEPS-1):
    model.AddImplication(goal_reached[t], goal_reached[t+1])
    model.Add(x[t+1] == goal[0]).OnlyEnforceIf(goal_reached[t])
    model.Add(y[t+1] == goal[1]).OnlyEnforceIf(goal_reached[t])

for t in range(MAX_STEPS-1):
    move_bools = []
    for dx, dy in moves:
        valid = model.NewBoolVar(f'move_{t}_{dx}_{dy}')
        move_bools.append(valid)
        model.Add(x[t+1] == x[t] + dx).OnlyEnforceIf(valid)
        model.Add(y[t+1] == y[t] + dy).OnlyEnforceIf(valid)
        model.Add(step_used[t+1] == 1).OnlyEnforceIf(valid)
        model.Add(step_used[t] == 1).OnlyEnforceIf(valid)
        model.Add(goal_reached[t] == 0).OnlyEnforceIf(valid)
    model.AddExactlyOne(move_bools + [goal_reached[t]]).OnlyEnforceIf(step_used[t])
    model.Add(x[t+1] == x[t]).OnlyEnforceIf(step_used[t+1].Not())
    model.Add(y[t+1] == y[t]).OnlyEnforceIf(step_used[t+1].Not())

for t in range(MAX_STEPS):
    model.Add(x[t] >= 0)
    model.Add(x[t] <= 4)
    model.Add(y[t] >= 0)
    model.Add(y[t] <= 4)
    # No obstacles in this example, but if grid[i][j]==1, forbid (i,j)
    # for i in range(5):
    #     for j in range(5):
    #         if grid[i][j] == 1:
    #             model.Add((x[t] != i) | (y[t] != j))

model.Minimize(sum(step_used))

solver = cp_model.CpSolver()
status = solver.Solve(model)

if status in [cp_model.OPTIMAL, cp_model.FEASIBLE]:
    path = []
    for t in range(MAX_STEPS):
        if solver.Value(step_used[t]):
            px = solver.Value(x[t])
            py = solver.Value(y[t])
            path.append((px, py))
            if (px, py) == goal:
                break
    print("Shortest diagonal path:", path)
    print("Number of steps:", len(path)-1)
    print("Total cost:", round((len(path)-1) * math.sqrt(2), 3))
else:
    print("No path found.")

Shortest diagonal path: [(1, 1), (2, 2), (3, 3), (4, 2), (3, 3), (4, 4)]
Number of steps: 5
Total cost: 7.071


In [2]:
# ------------------- TASK 02 ------------------- #

from ortools.sat.python import cp_model

grid = [
    [0, 1, 0, 0, 0],
    [1, 1, 1, 0, 0],
    [0, 1, 0, 0, 1],
    [0, 0, 0, 1, 1],
]

rows = len(grid)
cols = len(grid[0])

model = cp_model.CpModel()

x = []
for i in range(rows):
    row = []
    for j in range(cols):
        row.append(model.NewBoolVar(f'x_{i}_{j}'))
    x.append(row)

for i in range(rows):
    for j in range(cols):
        if grid[i][j] == 0:
            model.Add(x[i][j] == 0)

def neighbors(i, j):
    for di, dj in [(-1,0), (1,0), (0,-1), (0,1)]:
        ni, nj = i+di, j+dj
        if 0 <= ni < rows and 0 <= nj < cols:
            yield ni, nj

land_cells = [(i, j) for i in range(rows) for j in range(cols) if grid[i][j] == 1]
if not land_cells:
    print("No land present.")
else:
    root = land_cells[0]
    flow = {}
    for i, j in land_cells:
        for ni, nj in neighbors(i, j):
            if grid[ni][nj] == 1:
                flow[(i, j, ni, nj)] = model.NewIntVar(0, rows*cols, f'flow_{i}_{j}_{ni}_{nj}')
    for i, j in land_cells:
        if (i, j) == root:
            continue
        inflow = []
        outflow = []
        for ni, nj in neighbors(i, j):
            if grid[ni][nj] == 1:
                if (ni, nj, i, j) in flow:
                    inflow.append(flow[(ni, nj, i, j)])
                if (i, j, ni, nj) in flow:
                    outflow.append(flow[(i, j, ni, nj)])
        model.Add(sum(inflow) - sum(outflow) == x[i][j])
    inflow = []
    outflow = []
    for ni, nj in neighbors(root[0], root[1]):
        if grid[ni][nj] == 1:
            if (ni, nj, root[0], root[1]) in flow:
                inflow.append(flow[(ni, nj, root[0], root[1])])
            if (root[0], root[1], ni, nj) in flow:
                outflow.append(flow[(root[0], root[1], ni, nj)])
    model.Add(sum(outflow) - sum(inflow) == sum(x[i][j] for i, j in land_cells) - 1)
    for (i, j, ni, nj), f in flow.items():
        model.Add(f <= x[i][j] * rows * cols)
        model.Add(f <= x[ni][nj] * rows * cols)

    model.Maximize(sum(x[i][j] for i, j in land_cells))

    solver = cp_model.CpSolver()
    status = solver.Solve(model)

    if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        landmass = [[solver.Value(x[i][j]) for j in range(cols)] for i in range(rows)]
        perimeter = 0
        for i in range(rows):
            for j in range(cols):
                if landmass[i][j]:
                    for di, dj in [(-1,0), (1,0), (0,-1), (0,1)]:
                        ni, nj = i+di, j+dj
                        if not (0 <= ni < rows and 0 <= nj < cols) or not landmass[ni][nj]:
                            perimeter += 1
        print("Largest landmass:")
        for row in landmass:
            print(row)
        print(f"Perimeter: {perimeter}")
    else:
        print("No solution found.")


Largest landmass:
[0, 1, 0, 0, 0]
[1, 1, 1, 0, 0]
[0, 1, 0, 0, 0]
[0, 0, 0, 0, 0]
Perimeter: 12


In [3]:
# ------------------- TASK 03 ------------------- #


from ortools.constraint_solver import pywrapcp, routing_enums_pb2
import numpy as np

np.random.seed(42)
num_cities = 10
coords = np.random.rand(num_cities, 2) * 100 

def compute_euclidean_distance_matrix(locations):
	size = len(locations)
	matrix = {}
	for from_node in range(size):
		matrix[from_node] = {}
		for to_node in range(size):
			if from_node == to_node:
				matrix[from_node][to_node] = 0
			else:
				matrix[from_node][to_node] = int(
					np.linalg.norm(locations[from_node] - locations[to_node])
				)
	return matrix

distance_matrix = compute_euclidean_distance_matrix(coords)

manager = pywrapcp.RoutingIndexManager(num_cities, 1, 0)
routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
	from_node = manager.IndexToNode(from_index)
	to_node = manager.IndexToNode(to_index)
	return distance_matrix[from_node][to_node]

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
	routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)

solution = routing.SolveWithParameters(search_parameters)

def print_solution(manager, routing, solution):
	print("Optimal path for 10 cities:")
	index = routing.Start(0)
	route = []
	route_distance = 0
	while not routing.IsEnd(index):
		node = manager.IndexToNode(index)
		route.append(node)
		previous_index = index
		index = solution.Value(routing.NextVar(index))
		route_distance += routing.GetArcCostForVehicle(previous_index, index, 0)
	route.append(manager.IndexToNode(index))  # return to start
	print(" -> ".join(str(city) for city in route))
	print(f"Total distance: {route_distance}")

if solution:
	print_solution(manager, routing, solution)
else:
	print("No solution found.")


Optimal path for 10 cities:
0 -> 5 -> 3 -> 8 -> 7 -> 2 -> 9 -> 6 -> 1 -> 4 -> 0
Total distance: 286
